# Power Capacity Expansion Planning Under Uncertainty

This notebook accompanies the toy capacity expansion planning example and illustrates how a two-stage stochastic model is built and solved with `mpi-sppy`.

## The model

We consider a simple copper-plate planning model with no network and no unit commitment. The first-stage decisions are investments in new capacity:

$$
\min_x\; c^{\mathrm{inv}}(x) + \mathbb{E}_{\omega}[Q(x,\omega)]
$$

where the first-stage decision vector is $x = (x^{gas}, x^{wind}, x^{solar})$.
$x^{gas}$ represents the number of gas units of 50MW size each, $x^{wind}$ and $x^{solar}$ represent the installed capacity in MW.

**Construction limits.** Gas builds are capped at 12 blocks of 50 MW each, and wind and solar builds have explicit maximum MW limits.
$$
0 \le x^{gas} \le 12,\quad 0 \le x^{wind} \le 500,\quad 0 \le x^{solar} \le 450.
$$

The recourse term is the expected operating cost: dispatch from existing generators, dispatch from new capacity, and load shedding. For each scenario $\omega$ with demand $d_{h,\omega}$ and availability $a_{g,h,\omega}$,

$$
Q(x,\omega) = \min_y \sum_{h \in H} w_h \left(\sum_{g \in G} c_g y_{g,h,\omega} + c^{gas} y^{gas}_{h,\omega} + c^{wind} y^{wind}_{h,\omega} + c^{solar} y^{solar}_{h,\omega} + c^{shed} s_{h,\omega}\right)
$$
subject to:

**Power balance.** Supply plus load shedding must meet demand in every hour and scenario.
$$
\sum_{g \in G} y_{g,h,\omega} + y^{gas}_{h,\omega} + y^{wind}_{h,\omega} + y^{solar}_{h,\omega} + s_{h,\omega} = d_{h,\omega} \qquad \forall h,\omega,
$$
**Existing generator limits.** Each existing unit is limited by its nameplate capacity and scenario-dependent availability.
$$
0 \le y_{g,h,\omega} \le \bar{c}_g a_{g,h,\omega} \qquad \forall g,h,\omega,
$$
**New build dispatch limits.** New gas is limited by built blocks of 50 MW each, and wind and solar are limited by their built MW.
$$
0 \le y^{gas}_{h,\omega} \le 50 \cdot x^{gas} a^{gas}_{h,\omega},\quad 0 \le y^{wind}_{h,\omega} \le x^{wind} a^{wind}_{h,\omega},\quad 0 \le y^{solar}_{h,\omega} \le x^{solar} a^{solar}_{h,\omega} \qquad \forall h,\omega.
$$
The gas build decision is integer in 50 MW blocks, while wind and solar builds are continuous.

### Deterministic equivalent

Replacing the expectation with a finite scenario sum gives

$$
\min_{x, y_\omega}\; c^{\mathrm{inv}}(x) + \sum_{\omega \in \Omega} p_\omega Q(x,\omega)
$$


### Scenario-based extensive form

The equivalent extensive-form model writes a separate copy of the first-stage design variables for each scenario and enforces non-anticipativity explicitly.

Let $x_\omega = (x^{gas}_\omega, x^{wind}_\omega, x^{solar}_\omega)$ denote the investment decision in scenario $\omega$. Then the full formulation is

$$
\min_{\{x_\omega, y_\omega\}} \sum_{\omega \in \Omega} p_\omega \left[ c^{\mathrm{inv}}(x_\omega) + \sum_{h \in H} w_h \left(\sum_{g \in G} c_g y_{g,h,\omega} + c^{gas} y^{gas}_{h,\omega} + c^{wind} y^{wind}_{h,\omega} + c^{solar} y^{solar}_{h,\omega} + c^{shed} s_{h,\omega} \right) \right]
$$
subject to, for every scenario $\omega \in \Omega$:

**Investment limits.**
$$
0 \le x^{gas}_\omega \le 12,\quad 0 \le x^{wind}_\omega \le 500,\quad 0 \le x^{solar}_\omega \le 450.
$$
**Power balance.**
$$
\sum_{g \in G} y_{g,h,\omega} + y^{gas}_{h,\omega} + y^{wind}_{h,\omega} + y^{solar}_{h,\omega} + s_{h,\omega} = d_{h,\omega} \qquad \forall h \in H.
$$
**Existing generator limits.**
$$
0 \le y_{g,h,\omega} \le \bar{c}_g a_{g,h,\omega} \qquad \forall g \in G,\; h \in H.
$$
**New build dispatch limits.**
$$
0 \le y^{gas}_{h,\omega} \le 50 \cdot x^{gas}_\omega a^{gas}_{h,\omega},\quad 0 \le y^{wind}_{h,\omega} \le x^{wind}_\omega a^{wind}_{h,\omega},\quad 0 \le y^{solar}_{h,\omega} \le x^{solar}_\omega a^{solar}_{h,\omega} \qquad \forall h \in H.
$$
**Non-anticipativity.** All scenarios must choose the same investment plan, so
$$
x_\omega = x_{\omega'} \qquad \forall \omega,\omega' \in \Omega.
$$
The gas investment remains integer-valued in 50 MW blocks, while wind and solar investments are continuous.

## The toy-size system

The next cell loads a helper module where the Pyomo model is defined and plots the data of our toy-sized problem. 

In [ ]:
import json

import matplotlib.pyplot as plt
import notebook_analysis as na
import notebook_plots as npz

with open('system_data.json', 'r', encoding='utf-8') as f:
    system_data = json.load(f)
existing_df = na.solio.build_existing_capacity_pie(system_data)
demand_df = na.compute_scenario_data_table(system_data, num_scens=3, seed=0)
scenario_labels = ['High Demand', 'Medium Demand', 'Low Demand'] if len(demand_df) and len({row['scenario'] for row in demand_df}) == 3 else None
fig, axs = plt.subplots(1, 2, figsize=(14, 5))
npz.plot_existing_system_pie(axs[0], existing_df)
npz.plot_demand_by_scenario(axs[1], demand_df, system_data['representative_hours'], scenario_labels=scenario_labels)
plt.tight_layout()
plt.show()

## Solving in extensive form

The next cell runs the extensive form and writes solution output for later analysis.

In [ ]:
%%sh
mkdir -p _nb_outputs
python -m mpisppy.generic_cylinders --module-name power_cep --num-scens 3 --EF --EF-solver-name highs --tee-EF --solution-base-name _nb_outputs/ef_solution

## The expected value problem and the value of stochastic optimization

We now compare the expected-value (EV) solution with the stochastic recourse problem (RP). The key question is whether a single first-stage design chosen under expected conditions performs well once uncertainty is revealed.

We show the EV and RP new-build mix, then compare the EV-evaluated cost (EEV) with the RP optimum. We also compare RP vs EEV hourly production, including load shedding, using stacked bars.

In [ ]:
import json

import matplotlib.pyplot as plt
import notebook_analysis as na
import notebook_plots as npz

with open('system_data.json', 'r', encoding='utf-8') as f:
    system_data = json.load(f)
ev = na.solve_ev(system_data, num_scens=3, seed=0, solver_name='highs')
rp = na.load_rp_solution('_nb_outputs/ef_solution', system_data)
fig, axs = plt.subplots(1, 2, figsize=(12, 5))
npz.plot_build_pies(axs[0], axs[1], ev['builds_mw'], rp['builds_mw'])
plt.tight_layout()
plt.show()
print('EV objective:', ev['objective'])
print('RP builds (blocks):', rp['builds'])
print('RP builds (MW):', rp['builds_mw'])


In [ ]:
import json

import matplotlib.pyplot as plt
import notebook_analysis as na
import notebook_plots as npz

with open('system_data.json', 'r', encoding='utf-8') as f:
    system_data = json.load(f)
ev = na.solve_ev(system_data, num_scens=3, seed=0, solver_name='highs')
res = na.solve_ws_and_eev(system_data, ev['builds'], num_scens=3, seed=0, solver_name='highs')
rp = na.solve_rp(system_data, num_scens=3, seed=0, solver_name='highs')
hours = system_data['representative_hours']
scenario_order = [row['scenario'] for row in res['ws_rows']]
scenario_labels = ['High Demand', 'Medium Demand', 'Low Demand'] if len(scenario_order) == 3 else scenario_order
rp_df = na.generation_rows_to_dataframe(na.build_rp_generation_table(system_data, '_nb_outputs/ef_solution'))
eev_df = na.generation_rows_to_dataframe(res['eev_generation_rows'])
fig, ax_grid = plt.subplots(len(scenario_order), 2, figsize=(16, 4 * len(scenario_order)), sharex=True, sharey=True)
npz.plot_generation_comparison_figure(fig, ax_grid, rp_df, eev_df, hours, scenario_order, scenario_labels)
fig.suptitle('RP vs EEV hourly production by scenario')
plt.show()
print('WS expected cost:', res['ws_cost'])
print('EEV expected cost:', res['eev_cost'])
print('VSS = EEV - RP:', res['eev_cost'] - rp['objective'])
print('EVPI = RP - WS:', rp['objective'] - res['ws_cost'])


## Decomposition: PH

A generic two-stage stochastic program duplicates the first-stage variables across scenarios and then enforces non-anticipativity by making the scenario copies equal. PH relaxes those equality constraints and coordinates the copies iteratively.

### PH without spoke cylinders

The next command uses the PH hub alone, without spokes. In that mode, the hub does not get the same best-solution/best-bound progression that a full cylinder run provides.

In [ ]:
%%sh
python -m mpisppy.generic_cylinders --module-name power_cep --num-scens 3 --solver-name highs --default-rho 2.0 --linearize-proximal-terms --max-iterations 20 --rel-gap 1e-4 --intra-hub-conv-thresh 1e-6

### PH with cylinders
#### Default value of rho
Below, we use spoke cylinders to obtain provable upper and lower bounds of the solution obtained. Note that although the run is identical to the one we had before, we now obtain bounds on the quality of the solution **during execution**.

In [ ]:
%%sh
mpiexec -n 6 python -m mpi4py -m mpisppy.generic_cylinders --module-name power_cep --num-scens 3 --solver-name highs --default-rho 2.0 --lagrangian --xhatshuffle --max-iterations 50 --rel-gap 1e-4 --intra-hub-conv-thresh 1e-6 --linearize-proximal-terms 

#### Using a more sophisticated $\rho$-setting rule
Below, we use the `--sep-rho` $\rho$-setting rule.
Note the optimality gap is closed much more quickly. `mpi-sppy` already supports many features known to improve the performance of PH-based decomposition.
Refer to the [documentation](https://mpi-sppy.readthedocs.io/) for details.

In [ ]:
%%sh
mpiexec -n 6 python -m mpi4py -m mpisppy.generic_cylinders --module-name power_cep --num-scens 3 --solver-name highs --sep-rho --lagrangian --xhatshuffle --max-iterations 50 --rel-gap 1e-4 --intra-hub-conv-thresh 1e-6 --linearize-proximal-terms 